In [ ]:
# count the total number

import requests
from tqdm import tqdm

BASE = "https://hacker-news.firebaseio.com/v0"

max_id = requests.get(f"{BASE}/maxitem.json").json()

counts = {"story": 0, "comment": 0, "job": 0, "poll": 0, "other": 0}

for i in tqdm(range(max_id - 10000, max_id)):  # sample last 10k
    item = requests.get(f"{BASE}/item/{i}.json").json()
    if item:
        t = item.get("type", "other")
        counts[t] = counts.get(t, 0) + 1

print(counts)

In [1]:
import requests
import json
from tqdm import tqdm

BASE = "https://hacker-news.firebaseio.com/v0"

# Step 1: get top story IDs
ids = requests.get(f"{BASE}/topstories.json").json()

data = []

# Step 2: fetch first N stories
for i in tqdm(ids[:100]):   # change 100 to whatever you want
    item = requests.get(f"{BASE}/item/{i}.json").json()
    if item:
        data.append(item)

# Step 3: save
with open("hn_topstories.json", "w") as f:
    json.dump(data, f, indent=2)

100%|██████████| 100/100 [00:12<00:00,  8.31it/s]


In [1]:
import json

with open("hn_topstories.json") as f:
    data = json.load(f)

print(type(data))          # list or dict?
print(len(data))           # how many items

print(data[0])             # first example

<class 'list'>
100
{'by': 'sam-bee', 'descendants': 109, 'id': 47673360, 'kids': [47676122, 47675838, 47674344, 47675961, 47676243, 47675248, 47676382, 47675489, 47675046, 47676408, 47676013, 47676345, 47676187, 47673822, 47673777, 47674123, 47675086, 47676124, 47676231, 47675880, 47676191, 47676407, 47676001, 47675722, 47675782, 47675789, 47675939, 47675178, 47676093, 47674191, 47675453, 47675476, 47676077, 47676314, 47675947, 47675306, 47673361, 47674688, 47675542, 47675235, 47673886, 47675137, 47674931, 47674979, 47676476, 47673634], 'score': 232, 'time': 1775560064, 'title': 'Show HN: Brutalist Concrete Laptop Stand (2024)', 'type': 'story', 'url': 'https://sam-burns.com/posts/concrete-laptop-stand/'}


In [2]:
print(data[1])  

{'by': 'henrygarner', 'descendants': 120, 'id': 47673005, 'kids': [47676360, 47674745, 47674969, 47673786, 47673350, 47674291, 47676390, 47676050, 47673818, 47673728, 47674690, 47674852], 'score': 215, 'time': 1775557512, 'title': 'We found an undocumented bug in the Apollo 11 guidance computer code', 'type': 'story', 'url': 'https://www.juxt.pro/blog/a-bug-on-the-dark-side-of-the-moon/'}


In [2]:
import json
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import tiktoken
from tqdm import tqdm

# -----------------------------
# Config
# -----------------------------
INPUT_FILE = "hn_topstories.json"
SOURCE_NAME = "HackerNews_topstories"

# tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# -----------------------------
# Utils
# -----------------------------
def get_year(ts):
    try:
        return datetime.utcfromtimestamp(ts).year
    except:
        return None

def clean_html(text):
    if not text:
        return ""
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)

def extract_text(item):
    # for your current dataset (mostly stories)
    return item.get("text") or item.get("title")

def count_tokens(text):
    return len(enc.encode(text))

# -----------------------------
# Load data
# -----------------------------
with open(INPUT_FILE) as f:
    data = json.load(f)

print(f"Loaded {len(data)} raw items")

# -----------------------------
# Process
# -----------------------------
rows = []

for item in tqdm(data):
    text = extract_text(item)
    if not text:
        continue

    year = get_year(item.get("time"))
    if year is None or year < 2000:
        continue

    text = clean_html(text)
    if not text.strip():
        continue

    rows.append({
        "Source": SOURCE_NAME,
        "Date": year,
        "Text": text,
        "Token_count": count_tokens(text)
    })

df = pd.DataFrame(rows)

# -----------------------------
# Basic sanity checks
# -----------------------------
print("\n===== DATASET SUMMARY =====")
print(df.head())
print(f"\nTotal usable samples: {len(df)}")
print("\nYear distribution:")
print(df["Date"].value_counts().sort_index())

print("\nToken stats:")
print(df["Token_count"].describe())

# -----------------------------
# Save (pre-HF)
# -----------------------------
# df.to_csv("hn_clean.csv", index=False)
# df.to_json("hn_clean.json", orient="records", indent=2)

# print("\nSaved to:")
# print(" - hn_clean.csv")
# print(" - hn_clean.json")
df.to_csv("hn_clean.csv", index=False)  # optional backup
df.to_json("hn_clean.jsonl", orient="records", lines=True)

print("\nSaved to:")
print(" - hn_clean.csv")
print(" - hn_clean.jsonl")

Loaded 100 raw items


  0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_3283292/1251361032.py:22: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return datetime.utcfromtimestamp(ts).year
100%|██████████| 100/100 [00:00<00:00, 13286.15it/s]


===== DATASET SUMMARY =====
                  Source  Date  \
0  HackerNews_topstories  2026   
1  HackerNews_topstories  2026   
2  HackerNews_topstories  2026   
3  HackerNews_topstories  2026   
4  HackerNews_topstories  2026   

                                                Text  Token_count  
0    Show HN: Brutalist Concrete Laptop Stand (2024)           13  
1  We found an undocumented bug in the Apollo 11 ...           13  
2  Show HN: A cartographer's attempt to realistic...           15  
3                  Dropping Cloudflare for Bunny.net            7  
4                            Every GPU That Mattered            5  

Total usable samples: 100

Year distribution:
Date
2026    100
Name: count, dtype: int64

Token stats:
count    100.000000
mean      45.250000
std      122.425436
min        4.000000
25%        8.750000
50%       13.000000
75%       16.000000
max      775.000000
Name: Token_count, dtype: float64

Saved to:
 - hn_clean.csv
 - hn_clean.jsonl
